# EDA exploration (Pokémon VGC aggregates)

Interactive companion of `python -m python.analytics.eda_report`. The report generates the fixed findings; this notebook is for asking
your own questions on the same aggregate tables. It reuses the report's functions, so numbers cannot differ from `docs/EDA_FINDINGS.md`.

Run from the repository root after the aggregate modules have written `data/aggregates/`. Every table states its population
(regulation, rating band) and sample size; read rates together with them (see `METRICS.md`).

**Sections:** 0 reading rule (how much to trust a rate) · 1 usage by month · 2 strong vs weak players · 3 popularity vs win rate inside bands ·
4 Tera · 5 teammates (co-occurrence, then win rate against expectation) · 6 matchups (then robustness across rating bands and noise) ·
7 change between regulations · 8 does a result hold on later months · 9 what these tables cannot answer yet.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from python.analytics import eda_report as eda
from python.analytics.matchup_stats import opponents_of
from python.analytics.partner_stats import partners_of
from python.analytics.pokemon_usage import wilson
from python.analytics.usage_trend import species_trend

AGG = Path(os.environ.get("AGG_DIR", "data/aggregates"))
usage = eda.read_table(AGG, "pokemon_usage_stats")
monthly = eda.read_table(AGG, "pokemon_usage_monthly")
tera_types = eda.read_table(AGG, "pokemon_tera_types")
partner = eda.read_table(AGG, "pokemon_partner_stats")
matchup = eda.read_table(AGG, "pokemon_matchup_stats")
print({name: (None if t is None else len(t)) for name, t in
       [("usage", usage), ("monthly", monthly), ("tera_types", tera_types), ("partner", partner), ("matchup", matchup)]})

## Parameters
Change these values and re-run the cells below. `MIN_N` is the smallest number of teams / battles an estimate may rest on to be shown.

In [ ]:
sizes = usage[usage["rating_band"] == "all"].drop_duplicates("regulation").set_index("regulation")["player_battles"]
REGULATION = sizes.idxmax()     # the regulation with the most player-battles; e.g. "gen9vgc2025regi"
SPECIES = "Incineroar"
BAND = "all"                    # all, below_1200, 1200_1399, 1400_1499, 1500_plus
MIN_N = int(os.environ.get("MIN_N", 500))
print(REGULATION, SPECIES, BAND, f"MIN_N={MIN_N}", f"({int(sizes[REGULATION]):,} player-battles)")

## 0. Reading rule: how much to trust a rate
A win rate is an estimate. Two tools are used below: the 95% Wilson interval (already stored as `win_ci_low` / `win_ci_high`) and shrinkage
toward the baseline, which pulls small samples back (`PRIOR` = how many average observations are added). A rate whose interval contains
the baseline (50% or the band baseline) is not evidence of anything.

In [ ]:
PRIOR = 200


def ci_excludes(df, value, low="win_ci_low", high="win_ci_high"):
    """True where the 95% interval lies entirely above or below `value`."""
    return (df[low] > value) | (df[high] < value)


def shrink(wins, n, base=0.5, k=PRIOR):
    return (wins + k * base) / (n + k)


demo = pd.DataFrame({"wins": [30, 5400], "n": [50, 10_000]})
lo, hi = wilson(demo["wins"], demo["n"])
demo["raw_%"] = 100 * demo["wins"] / demo["n"]
demo["ci95_%"] = [f"{100 * a:.1f} to {100 * b:.1f}" for a, b in zip(lo, hi)]
demo["shrunk_%"] = (100 * shrink(demo["wins"], demo["n"])).round(1)
display(demo)     # 60% from 50 games: interval contains 50%. 54% from 10,000 games: it does not.

## 1. How does a species' usage move month by month?
Months flagged `tail` (the format is ending) are shown in the table but left out of the chart.

In [ ]:
t = species_trend(monthly, SPECIES, REGULATION)
display(t)
ok = t[~t["tail"]]
if len(ok):
    ax = ok.plot(x="month", y="usage_%", marker="o", legend=False, title=f"{SPECIES}: usage by month, {REGULATION}")
    ax.set_ylabel("% of teams")
    plt.show()

## 2. What do strong players pick differently?
Usage at 1500+ minus usage below 1200 (percentage points), species with at least 2,000 teams in both bands.

In [ ]:
display(eda.band_shift(usage, REGULATION, n=10))

## 3. Does popularity predict win rate, inside each rating band?
Spearman correlation of usage and win rate relative to the band's own baseline, one row per regulation. The species set is fixed per regulation (species with at least 500 teams at 1500+; `species` counts them) so the bands are comparable. A value that shrinks from the weak to the strong bands would point to a rating effect.

In [ ]:
display(eda.popularity_by_band(usage))

## 4. Which Tera types does the species use?

In [ ]:
tt = tera_types[(tera_types["regulation"] == REGULATION) & (tera_types["species"] == SPECIES) & (tera_types["rating_band"] == BAND)]
tt = tt.sort_values("events", ascending=False)
display(tt[["tera_type", "events", "type_share", "wins"]])
if len(tt):
    tt.plot.bar(x="tera_type", y="type_share", legend=False, title=f"Tera types of {SPECIES}, {REGULATION}")
    plt.show()

## 5. Which teammates does it appear with?
`lift` is co-occurrence compared with chance, not proof of synergy.

In [ ]:
display(partners_of(partner, SPECIES, REGULATION, BAND))

### 5b. Which pairs win more than their members' own strength predicts?
A pair of strong species wins a lot without helping each other. Rough screen: expected pair win rate = band baseline + each member's own
win rate above baseline (additive; teams overlap, so this is a heuristic, not a model). `excess_pp` = observed minus expected; `z` uses the
expected rate. A real test of cores needs a model on team composition (section 9). Only pairs on at least `MIN_N` teams.

In [ ]:
sp = eda._species_level(usage[usage["regulation"] == REGULATION], BAND).set_index("species")
sp_rate = sp["wins"] / sp["appearances"]
pairs = partner[(partner["regulation"] == REGULATION) & (partner["rating_band"] == BAND) & (partner["teams_both"] >= MIN_N)].copy()
base = pairs["band_win_rate"]
pairs["expected"] = (base + (pairs["species_a"].map(sp_rate) - base) + (pairs["species_b"].map(sp_rate) - base)).clip(0.01, 0.99)
pairs["excess_pp"] = 100 * (pairs["win_rate"] - pairs["expected"])
pairs["z"] = (pairs["win_rate"] - pairs["expected"]) / np.sqrt(pairs["expected"] * (1 - pairs["expected"]) / pairs["teams_both"])
pairs = pairs.dropna(subset=["z"]).sort_values("z", ascending=False)
view = pairs.assign(**{"win_%": 100 * pairs["win_rate"], "expected_%": 100 * pairs["expected"]})
cols = ["species_a", "species_b", "teams_both", "lift", "win_%", "expected_%", "excess_pp", "z"]
if len(view):
    print(f"{len(view):,} pairs; above expectation:"); display(view[cols].head(10).round(2))
    print("below expectation:"); display(view[cols].tail(5).iloc[::-1].round(2))
else:
    print("no pair reaches MIN_N teams in this regulation and band")

## 6. Which opponent teams do teams with it beat or lose to?
Ranked by z-score against what both teams' overall strength predicts (log5); small effects are normal. Only opponents with at least `MIN_N` battles.

In [ ]:
hard, easy = opponents_of(matchup, SPECIES, REGULATION, BAND, min_battles=MIN_N)
print("hardest"); display(hard)
print("easiest"); display(easy)

### 6b. Is a counter a high-rating counter or a low-rating one, and how much of this is noise?
For the hardest opponents (band `all`), the same matchup per rating band: `vs_expected` in pp and the battles behind it (the band is the band of
the player of `SPECIES`). The second cell counts how many matchups of the regulation exceed what chance alone would give: under pure noise
about 4.6% have |z| > 2 and almost none |z| > 4. Teams of one player and Bo3 games are not independent, so real z-scores are smaller than shown.

In [ ]:
hard_all, _ = opponents_of(matchup, SPECIES, REGULATION, "all", min_battles=MIN_N)
rows = matchup[(matchup["species_a"] == SPECIES) & (matchup["regulation"] == REGULATION) & matchup["species_b"].isin(hard_all["opponent"])]
if len(rows):
    pp = (100 * rows.pivot_table(index="species_b", columns="rating_band", values="win_rate_vs_expected")).round(1).add_prefix("pp_")
    n = rows.pivot_table(index="species_b", columns="rating_band", values="battles").add_prefix("n_")
    display(pp.join(n).reindex(hard_all["opponent"]))
else:
    print("no opponent reaches MIN_N battles")

In [ ]:
a = matchup[(matchup["regulation"] == REGULATION) & (matchup["rating_band"] == "all") & (matchup["battles"] >= MIN_N)]
if len(a):
    z = a["z_vs_expected"].abs()
    print(f"{len(a):,} ordered pairs with >= {MIN_N} battles: |z| > 2 in {100 * (z > 2).mean():.1f}%, |z| > 4 in {100 * (z > 4).mean():.1f}%; "
          f"median |deviation| {100 * a['win_rate_vs_expected'].abs().median():.2f} pp")

## 7. How does the meta change between regulations?
Species usage and win rate above baseline in the regulation before `REGULATION` versus `REGULATION`, **tail months excluded** (so a fading format does
not distort either side). Population: band `all`. Caution: each regulation has its own legal species, so a change in usage can come from the rules, not from players.
Risers and fallers need at least `MIN_N` teams in both; newcomers are species absent from the earlier regulation.

In [ ]:
dim = eda.read_table(AGG, "dim_regulation")
order = list(sizes.index) if dim is None else list(dim.sort_values("sort_order")["regulation"])
order = [r for r in order if r in sizes.index]
big = [r for r in order if sizes[r] >= 50_000]
order = big if REGULATION in big and len(big) >= 2 else order
PREV = order[order.index(REGULATION) - 1] if REGULATION in order and order.index(REGULATION) > 0 else None


def core_usage(reg):
    m = monthly[(monthly["regulation"] == reg) & (~monthly["tail"].astype(bool))]
    g = m.groupby(["species_id", "species"], as_index=False)[["appearances", "wins"]].sum()
    pb = m.drop_duplicates("month")["player_battles"].sum()
    if g.empty or not pb:
        return g.assign(usage=np.nan, win_delta=np.nan)
    g["usage"] = g["appearances"] / pb
    g["win_delta"] = g["wins"] / g["appearances"] - g["wins"].sum() / g["appearances"].sum()
    return g


a, b = (core_usage(PREV), core_usage(REGULATION)) if PREV else (None, None)
if PREV is None or a.empty or b.empty:
    print("no earlier regulation with data to compare against", REGULATION)
else:
    print(f"{PREV} -> {REGULATION}")
    m = a.merge(b, on=["species_id", "species"], how="outer", suffixes=("_prev", "_now"))
    m["change_pp"] = 100 * (m["usage_now"].fillna(0) - m["usage_prev"].fillna(0))
    both = m[(m["appearances_prev"].fillna(0) >= MIN_N) & (m["appearances_now"].fillna(0) >= MIN_N)].copy()
    both["win_change_pp"] = 100 * (both["win_delta_now"] - both["win_delta_prev"])
    show = both.assign(usage_prev=100 * both["usage_prev"], usage_now=100 * both["usage_now"], win_prev_pp=100 * both["win_delta_prev"],
                       win_now_pp=100 * both["win_delta_now"]).sort_values("change_pp")
    cols = ["species", "usage_prev", "usage_now", "change_pp", "win_prev_pp", "win_now_pp", "win_change_pp"]
    print("fallers:"); display(show[cols].head(8).round(1))
    print("risers:"); display(show[cols].tail(8).iloc[::-1].round(1))
    if len(both) >= 5:
        print("Spearman, usage change vs win-rate change:", round(both["change_pp"].corr(both["win_change_pp"], method="spearman"), 2))
    new = m[m["appearances_prev"].isna() & (m["appearances_now"].fillna(0) >= MIN_N)]
    new = new.assign(usage_now=100 * new["usage_now"], win_now_pp=100 * new["win_delta_now"]).sort_values("usage_now", ascending=False)
    print("newcomers:"); display(new[["species", "usage_now", "win_now_pp"]].head(8).round(1))

## 8. Does a result hold on later months?
A first out-of-time check on species win rates: the months of `REGULATION` (tail excluded) are split in two halves; species are ranked by win rate above
baseline in the first half and the same number is read in the second. Some fall back toward 0 by chance alone (regression to the mean); a result
worth using as a recommendation should keep its sign and a good part of its size. Pairs and matchups are not stored per month, so they cannot
be checked this way yet (section 9).

In [ ]:
mm = monthly[(monthly["regulation"] == REGULATION) & (~monthly["tail"].astype(bool)) & (monthly["month"] != "unknown")]
months = sorted(mm["month"].unique())
half = len(months) // 2


def half_rates(ms):
    g = mm[mm["month"].isin(ms)].groupby("species", as_index=False)[["appearances", "wins"]].sum()
    g["delta_pp"] = 100 * (g["wins"] / g["appearances"] - g["wins"].sum() / g["appearances"].sum())
    return g.set_index("species")


if half < 1:
    print("fewer than 2 usable months in", REGULATION)
else:
    early, late = half_rates(months[:half]), half_rates(months[half:])
    j = early.join(late, lsuffix="_early", rsuffix="_late", how="inner")
    j = j[(j["appearances_early"] >= MIN_N) & (j["appearances_late"] >= MIN_N)]
    print(f"early: {months[0]}..{months[half - 1]}, late: {months[half]}..{months[-1]}; {len(j)} species with >= {MIN_N} teams in both")
    if len(j) >= 5:
        print("Spearman early vs late win rate above baseline:", round(j["delta_pp_early"].corr(j["delta_pp_late"], method="spearman"), 2))
    j = j.sort_values("delta_pp_early", ascending=False)[["appearances_early", "delta_pp_early", "appearances_late", "delta_pp_late"]]
    display(pd.concat([j.head(8), j.tail(8)]).loc[lambda d: ~d.index.duplicated()].round(2))

## 9. What these tables cannot answer yet
Questions from the EDA review that need data not aggregated today. Each names the missing piece, so none is answered by guessing.

| Question | Missing | Next step |
|---|---|---|
| Which 3-Pokémon cores beat their parts? | no trio table (about 20 trios per team: must be pruned by support) | `partner_stats` variant for trios with a support cut-off, then a model on team composition |
| Which lead pair and back pair work against an archetype? | `battle_leads` (14.5M rows) has no aggregate | `lead_stats.py`: lead pair x win rate, per regulation and band |
| Which build (item, ability, moves) is better? | items and abilities are only revealed in some events (selection bias) | state the bias, then aggregate moves / revealed items per species |
| Do pair or matchup results hold out of sample? | partner and matchup tables are not split by month | add a month or half-regulation key, build on the first half, test on the second |
| Replace one team member with little loss | no recommender yet | Sprint 6, with the hold-out defined first |